# Amazon sales: pandas quality checks
This notebook uses the **first 500 source rows** as a small, non-representative practice sample. It removes order IDs, city and postal code. The goal is to show parsing, profiling, and cautious summary checks; it does not reproduce the full Power BI findings.


In [ ]:
from pathlib import Path
import pandas as pd
path = Path('sample-500-rows.csv')
raw = pd.read_csv(path, dtype='string', keep_default_na=False)
print(raw.shape)
raw.head()

In [ ]:
df = raw.copy()
df.columns = df.columns.str.lower().str.replace('-', '_')
df['order_date'] = pd.to_datetime(df['date'], format='%m-%d-%y', errors='coerce')
df['amount_num'] = pd.to_numeric(df['amount'], errors='coerce')
df['qty_num'] = pd.to_numeric(df['qty'], errors='coerce')
df['ship_state'] = df['ship_state'].str.strip().str.upper()
quality = pd.Series({'rows': len(df), 'invalid_dates': int(df['order_date'].isna().sum()), 'invalid_amounts': int(df['amount_num'].isna().sum()), 'duplicate_sample_rows': int(df.duplicated().sum())})
quality

In [ ]:
# Descriptive sample check only. Do not equate all status amounts with realized revenue.
status_summary = (df.groupby('status', dropna=False).agg(rows=('status', 'size'), stated_amount=('amount_num', 'sum')).sort_values('rows', ascending=False))
status_summary

## Limits
The sample is ordered as supplied, so it cannot support whole-dataset estimates. Missing amount and zero quantity may be legitimate for some statuses. Use the full project model and business definitions before calling a figure revenue, cancellation leakage, or profit.
